# People Extraction Pipeline for Google Colab

This notebook uploads one photo, installs Meta's native SAM3 repository, segments `people` with a text prompt, saves 2D mask artifacts, runs FLUX.1 Fill [dev] through Diffusers to fill the removed people regions, runs Apple SHARP to generate a 3D Gaussian splat `.ply`, and filters the splats by reprojecting them back into the source view, applying the foreground depth thresholds independently to each SAM3 person mask, and merging the retained splats into one people-only output.

## Runtime requirements

- Google Colab GPU runtime
- Hugging Face access to the gated `facebook/sam3` weights
- Hugging Face access to the gated `black-forest-labs/FLUX.1-Fill-dev` weights
- Internet access to install dependencies and download the SHARP, SAM3, and FLUX.1 Fill model assets on first run


In [ ]:
%%capture
%pip install -q --upgrade --force-reinstall --no-cache-dir "Pillow>=11.1,<12"
%pip install -q --upgrade --force-reinstall --no-cache-dir opencv-python-headless matplotlib plyfile
%pip install -q --upgrade --force-reinstall --no-cache-dir diffusers transformers accelerate safetensors sentencepiece protobuf
!if [ ! -d /content/sam3/.git ]; then git clone https://github.com/facebookresearch/sam3.git /content/sam3; fi
%cd /content/sam3
%pip install -q -e ".[notebooks]"
%cd /content
!if [ ! -d /content/ml-sharp/.git ]; then git clone https://github.com/apple/ml-sharp.git /content/ml-sharp; fi
%cd /content/ml-sharp
%pip install -q -r requirements.txt
%pip install -q -e .
%pip install -q --upgrade --force-reinstall --no-cache-dir "diffusers==0.36.0" "transformers>=4.46.0" "accelerate>=1.1.0" "huggingface_hub>=0.34,<1.0"
!python3 -c "import diffusers, huggingface_hub; print('diffusers', diffusers.__version__); print('huggingface_hub', huggingface_hub.__version__)"
!sharp --help > /dev/null
%cd /content


In [ ]:
from pathlib import Path
import shutil
import sys

import torch
import huggingface_hub

TEXT_PROMPT = "people"
FOREGROUND_NEAREST_PERCENTILE = 1.0
FOREGROUND_DEPTH_WINDOW = 0.7
SAM3_REPO_DIR = Path("/content/sam3")
SHARP_REPO_DIR = Path("/content/ml-sharp")
OUTPUT_DIR = Path("/content/people_pipeline_outputs")
SHARP_OUTPUT_DIR = OUTPUT_DIR / "sharp_output"
INPAINT_OUTPUT_DIR = OUTPUT_DIR / "flux_fill_output"
INPAINT_MODEL_ID = "black-forest-labs/FLUX.1-Fill-dev"
INPAINT_DEVICE = "cuda"
INPAINT_DTYPE = torch.bfloat16
INPAINT_OUTPUT_NAME = "people_removed_inpainted.png"
INPAINT_MASK_DILATION_PX = 25
INPAINT_PROMPT = "empty restaurant interior with the yellow dog statue unobstructed, booths, floor, and background visible, no people"
INPAINT_GUIDANCE_SCALE = 30.0
INPAINT_NUM_STEPS = 50
INPAINT_MAX_SEQUENCE_LENGTH = 512
INPAINT_SEED = 0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHARP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPAINT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for candidate in (SAM3_REPO_DIR, SAM3_REPO_DIR / "src"):
    if candidate.exists():
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required for this notebook. In Colab, switch to Runtime > Change runtime type > GPU and rerun."
    )

hf_hub_version = tuple(int(part) for part in huggingface_hub.__version__.split(".")[:3])
if hf_hub_version < (0, 34, 0):
    raise RuntimeError(
        "This notebook needs huggingface_hub>=0.34 for Diffusers FLUX Fill. Restart the Colab runtime, rerun the install cell, and confirm the printed versions show a recent diffusers and huggingface_hub."
    )

print(f"Python executable: {sys.executable}")
print(f"Torch version: {torch.__version__}")
print(f"huggingface_hub version: {huggingface_hub.__version__}")
print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Output directory: {OUTPUT_DIR}")
print(
    "Per-person foreground depth slice: for each SAM3 person mask, find the front masked depth at percentile "
    f"{FOREGROUND_NEAREST_PERCENTILE} and keep splats within {FOREGROUND_DEPTH_WINDOW} depth units behind it"
)
print(f"SAM3 repo directory: {SAM3_REPO_DIR}")
print(f"SHARP repo directory: {SHARP_REPO_DIR}")
print(f"FLUX output directory: {INPAINT_OUTPUT_DIR}")
print(f"FLUX model: {INPAINT_MODEL_ID}")
print(f"FLUX device/dtype: {INPAINT_DEVICE}/{INPAINT_DTYPE}")
print(f"FLUX mask dilation: {INPAINT_MASK_DILATION_PX} px")
print(f"FLUX prompt: {INPAINT_PROMPT}")


In [ ]:
from huggingface_hub import login

login()


In [ ]:
import subprocess
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
from diffusers import FluxFillPipeline
from PIL import Image, ImageOps
from plyfile import PlyData, PlyElement
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("This notebook is intended to run inside Google Colab.") from exc

_SAM3_CACHE: dict[str, Any] = {}
_INPAINT_CACHE: dict[str, Any] = {}


def load_uploaded_image(out_dir: Path) -> tuple[Image.Image, np.ndarray, Path]:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file was uploaded.")

    original_name, raw_bytes = next(iter(uploaded.items()))
    image_path = out_dir / Path(original_name).name
    image_path.write_bytes(raw_bytes)

    pil_image = Image.open(image_path)
    pil_image = ImageOps.exif_transpose(pil_image).convert("RGB")
    pil_image.save(image_path)

    np_image = np.array(pil_image)
    return pil_image, np_image, image_path


def get_sam3_components() -> Sam3Processor:
    cache_key = f"sam3:{torch.cuda.get_device_name(0)}"
    if cache_key not in _SAM3_CACHE:
        model = build_sam3_image_model().to("cuda")
        model.eval()
        processor = Sam3Processor(model)
        _SAM3_CACHE[cache_key] = processor
    return _SAM3_CACHE[cache_key]


def segment_people_sam3(
    image: Image.Image,
    text_prompt: str = TEXT_PROMPT,
) -> dict[str, Any]:
    processor = get_sam3_components()
    inference_state = processor.set_image(image)
    results = processor.set_text_prompt(state=inference_state, prompt=text_prompt)

    mask_list = results.get("masks", []) if results is not None else []
    people_mask = np.zeros((image.height, image.width), dtype=bool)
    instance_masks = []

    for mask in mask_list:
        if hasattr(mask, "detach"):
            mask_np = mask.detach().cpu().numpy()
        else:
            mask_np = np.asarray(mask)
        if mask_np.ndim > 2:
            mask_np = np.squeeze(mask_np)
        mask_bool = mask_np.astype(bool)
        if mask_bool.shape != people_mask.shape:
            raise ValueError(
                f"Unexpected SAM3 mask shape {mask_bool.shape}; expected {(image.height, image.width)}"
            )
        instance_masks.append(mask_bool)
        people_mask |= mask_bool

    return {
        "prompt": text_prompt,
        "results": results,
        "instance_masks": instance_masks,
        "people_mask": people_mask,
        "num_instances": len(instance_masks),
    }


def save_mask_artifacts(np_image: np.ndarray, people_mask: np.ndarray, out_dir: Path) -> dict[str, Path]:
    mask_path = out_dir / "people_mask.png"
    overlay_path = out_dir / "people_overlay.png"
    removed_path = out_dir / "people_removed_black.png"

    mask_image = people_mask.astype(np.uint8) * 255
    Image.fromarray(mask_image).save(mask_path)

    removed_image = np_image.copy()
    removed_image[people_mask] = 0
    Image.fromarray(removed_image).save(removed_path)

    overlay_image = np_image.copy().astype(np.float32)
    if people_mask.any():
        overlay_color = np.array([255, 64, 64], dtype=np.float32)
        alpha = 0.45
        overlay_image[people_mask] = (
            (1.0 - alpha) * overlay_image[people_mask] + alpha * overlay_color
        )
    overlay_image = np.clip(overlay_image, 0, 255).astype(np.uint8)
    Image.fromarray(overlay_image).save(overlay_path)

    return {
        "mask_path": mask_path,
        "overlay_path": overlay_path,
        "removed_path": removed_path,
    }


def get_flux_fill_pipeline() -> FluxFillPipeline:
    cache_key = f"flux-fill:{INPAINT_MODEL_ID}:{INPAINT_DEVICE}:{INPAINT_DTYPE}"
    if cache_key not in _INPAINT_CACHE:
        try:
            pipeline = FluxFillPipeline.from_pretrained(
                INPAINT_MODEL_ID,
                torch_dtype=INPAINT_DTYPE,
            )
        except Exception as exc:
            raise RuntimeError(
                "Failed to load black-forest-labs/FLUX.1-Fill-dev. Make sure you are logged into Hugging Face in Colab and have accepted the model access terms."
            ) from exc
        pipeline = pipeline.to(INPAINT_DEVICE)
        _INPAINT_CACHE[cache_key] = pipeline
    return _INPAINT_CACHE[cache_key]


def _round_down_to_multiple(value: int, multiple: int = 32) -> int:
    return max(multiple, value - (value % multiple))


def build_inpaint_mask(mask_path: Path, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    prepared_mask_path = out_dir / "people_mask_inpaint.png"
    mask_array = np.array(Image.open(mask_path).convert("L"))
    binary_mask = (mask_array > 0).astype(np.uint8) * 255
    kernel = np.ones((INPAINT_MASK_DILATION_PX, INPAINT_MASK_DILATION_PX), dtype=np.uint8)
    dilated_mask = cv2.dilate(binary_mask, kernel, iterations=1)
    Image.fromarray(dilated_mask).save(prepared_mask_path)
    return prepared_mask_path


def run_flux_fill_inpainting(image_path: Path, mask_path: Path, out_dir: Path, prompt: str) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    final_output_path = out_dir / INPAINT_OUTPUT_NAME
    original_image = Image.open(image_path).convert("RGB")
    original_mask = Image.open(mask_path).convert("L")
    image_size = original_image.size
    mask_size = original_mask.size
    if mask_size != image_size:
        raise ValueError(
            f"Inpaint mask size {mask_size} does not match image size {image_size}."
        )

    mask_array = np.array(original_mask)
    if not np.any(mask_array):
        shutil.copy2(image_path, final_output_path)
        return final_output_path

    prepared_mask_path = build_inpaint_mask(mask_path, out_dir)
    prepared_mask = Image.open(prepared_mask_path).convert("L")
    inference_width = _round_down_to_multiple(original_image.width)
    inference_height = _round_down_to_multiple(original_image.height)
    resized_image = original_image.resize((inference_width, inference_height), Image.Resampling.LANCZOS)
    resized_mask = prepared_mask.resize((inference_width, inference_height), Image.Resampling.NEAREST)
    pipeline = get_flux_fill_pipeline()
    generator = torch.Generator(device=INPAINT_DEVICE).manual_seed(INPAINT_SEED)

    try:
        result = pipeline(
            prompt=prompt,
            image=resized_image,
            mask_image=resized_mask,
            guidance_scale=INPAINT_GUIDANCE_SCALE,
            num_inference_steps=INPAINT_NUM_STEPS,
            max_sequence_length=INPAINT_MAX_SEQUENCE_LENGTH,
            height=inference_height,
            width=inference_width,
            generator=generator,
        ).images[0]
    except Exception as exc:
        raise RuntimeError(
            "FLUX.1 Fill inference failed. Confirm that you accepted the Hugging Face model terms for black-forest-labs/FLUX.1-Fill-dev and that your Colab GPU has enough memory."
        ) from exc

    if result.size != original_image.size:
        result = result.resize(original_image.size, Image.Resampling.LANCZOS)
    result.save(final_output_path)
    return final_output_path


def run_sharp(image_path: Path, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    sharp_executable = shutil.which("sharp")
    if sharp_executable is None:
        raise RuntimeError("The `sharp` CLI was not found after installation.")

    cmd = [
        sharp_executable,
        "predict",
        "-i",
        str(image_path),
        "-o",
        str(out_dir),
        "--device",
        "cuda",
    ]
    subprocess.run(cmd, check=True)

    ply_path = out_dir / f"{image_path.stem}.ply"
    if not ply_path.exists():
        raise FileNotFoundError(f"Expected SHARP output PLY at {ply_path}")
    return ply_path


def _extract_element_values(ply_data: PlyData, element_name: str, field_name: str) -> np.ndarray | None:
    for element in ply_data.elements:
        dtype_names = element.data.dtype.names or ()
        if element.name == element_name and field_name in dtype_names:
            return np.asarray(element.data[field_name])
    return None


def _parse_intrinsics_and_image_size(
    ply_data: PlyData,
    default_image_shape: tuple[int, int],
) -> tuple[np.ndarray, tuple[int, int]]:
    intrinsic_data = _extract_element_values(ply_data, "intrinsic", "intrinsic")
    image_size_data = _extract_element_values(ply_data, "image_size", "image_size")

    if intrinsic_data is None:
        height, width = default_image_shape
        intrinsic_3x3 = np.array(
            [
                [512.0, 0.0, width * 0.5],
                [0.0, 512.0, height * 0.5],
                [0.0, 0.0, 1.0],
            ],
            dtype=np.float32,
        )
        return intrinsic_3x3, (width, height)

    intrinsic_data = np.asarray(intrinsic_data).reshape(-1)
    if image_size_data is None:
        if intrinsic_data.size != 4:
            raise ValueError(
                "Expected legacy SHARP intrinsics with 4 values when image_size metadata is missing."
            )
        fx, fy, width, height = intrinsic_data.tolist()
        intrinsic_3x3 = np.array(
            [[fx, 0.0, width * 0.5], [0.0, fy, height * 0.5], [0.0, 0.0, 1.0]],
            dtype=np.float32,
        )
        return intrinsic_3x3, (int(width), int(height))

    if intrinsic_data.size != 9:
        raise ValueError(f"Expected 9 intrinsic values, received {intrinsic_data.size}.")

    intrinsic_3x3 = intrinsic_data.reshape(3, 3).astype(np.float32)
    image_size_data = np.asarray(image_size_data).reshape(-1)
    if image_size_data.size != 2:
        raise ValueError(f"Expected 2 image_size values, received {image_size_data.size}.")
    width, height = image_size_data.tolist()
    return intrinsic_3x3, (int(width), int(height))


def _parse_extrinsic_matrix(ply_data: PlyData) -> np.ndarray:
    extrinsic_data = _extract_element_values(ply_data, "extrinsic", "extrinsic")
    if extrinsic_data is None:
        return np.eye(4, dtype=np.float32)

    extrinsic_data = np.asarray(extrinsic_data).reshape(-1)
    extrinsic_matrix = np.eye(4, dtype=np.float32)
    if extrinsic_data.size == 12:
        extrinsic_matrix[:3] = extrinsic_data.reshape(3, 4)
        extrinsic_matrix[:3, :3] = extrinsic_matrix[:3, :3].T
        return extrinsic_matrix
    if extrinsic_data.size == 16:
        return extrinsic_data.reshape(4, 4).astype(np.float32)
    raise ValueError(f"Unrecognized extrinsic metadata length: {extrinsic_data.size}")


def filter_ply_with_masks(
    ply_path: Path,
    instance_masks: list[np.ndarray],
    image_shape: tuple[int, int],
    out_dir: Path,
    nearest_percentile: float = FOREGROUND_NEAREST_PERCENTILE,
    depth_window: float = FOREGROUND_DEPTH_WINDOW,
) -> tuple[Path, dict[str, Any]]:
    out_dir.mkdir(parents=True, exist_ok=True)
    filtered_path = out_dir / "people_only_gaussians.ply"

    expected_shape = tuple(image_shape)
    for index, mask in enumerate(instance_masks):
        if mask.shape != expected_shape:
            raise ValueError(
                f"Instance mask {index} shape {mask.shape} does not match expected {expected_shape}."
            )

    ply_data = PlyData.read(str(ply_path))
    vertex_element = next((element for element in ply_data.elements if element.name == "vertex"), None)
    if vertex_element is None:
        raise KeyError("PLY does not contain a vertex element.")

    vertex_data = vertex_element.data
    total_vertices = len(vertex_data)
    clamped_percentile = float(np.clip(nearest_percentile, 0.0, 100.0))
    instance_stats: list[dict[str, Any]] = []

    if total_vertices == 0:
        filtered_vertex_data = vertex_data.copy()
        kept_vertices = 0
        inside_image = 0
        inside_mask = 0
        depth_rejected = 0
        metadata_image_size = (expected_shape[1], expected_shape[0])
        for instance_index, person_mask in enumerate(instance_masks):
            instance_stats.append(
                {
                    "instance_index": int(instance_index),
                    "mask_pixels": int(person_mask.sum()),
                    "inside_image_vertices": 0,
                    "inside_mask_vertices": 0,
                    "depth_rejected_vertices": 0,
                    "kept_vertices": 0,
                    "front_depth": None,
                }
            )
    else:
        intrinsic_3x3, metadata_image_size = _parse_intrinsics_and_image_size(ply_data, expected_shape)
        extrinsic_4x4 = _parse_extrinsic_matrix(ply_data)

        xyz = np.stack(
            [
                np.asarray(vertex_data["x"], dtype=np.float32),
                np.asarray(vertex_data["y"], dtype=np.float32),
                np.asarray(vertex_data["z"], dtype=np.float32),
            ],
            axis=1,
        )
        homogeneous_xyz = np.concatenate(
            [xyz, np.ones((xyz.shape[0], 1), dtype=np.float32)],
            axis=1,
        )
        camera_xyz = (extrinsic_4x4 @ homogeneous_xyz.T).T[:, :3]
        positive_depth = camera_xyz[:, 2] > 0

        projected = (intrinsic_3x3 @ camera_xyz.T).T
        finite_projection = np.isfinite(projected).all(axis=1)
        uv = projected[:, :2] / projected[:, 2:3]
        finite_uv = np.isfinite(uv).all(axis=1)
        u = np.rint(uv[:, 0]).astype(np.int64)
        v = np.rint(uv[:, 1]).astype(np.int64)

        height, width = expected_shape
        inside_image_mask = (
            positive_depth
            & finite_projection
            & finite_uv
            & (u >= 0)
            & (u < width)
            & (v >= 0)
            & (v < height)
        )

        image_indices = np.flatnonzero(inside_image_mask)
        keep_mask = np.zeros(total_vertices, dtype=bool)
        any_mask_projection = np.zeros(total_vertices, dtype=bool)

        for instance_index, person_mask in enumerate(instance_masks):
            mask_projection = np.zeros(total_vertices, dtype=bool)
            mask_projection[image_indices] = person_mask[v[image_indices], u[image_indices]]
            valid_indices = np.flatnonzero(mask_projection)
            any_mask_projection |= mask_projection

            if valid_indices.size:
                masked_depths = camera_xyz[valid_indices, 2]
                front_depth = float(np.percentile(masked_depths, clamped_percentile))
                keep_valid = masked_depths <= front_depth + depth_window
                keep_indices = valid_indices[keep_valid]
                keep_mask[keep_indices] = True
                instance_depth_rejected = int(valid_indices.size - keep_valid.sum())
            else:
                front_depth = None
                keep_indices = np.empty(0, dtype=np.int64)
                instance_depth_rejected = 0

            instance_stats.append(
                {
                    "instance_index": int(instance_index),
                    "mask_pixels": int(person_mask.sum()),
                    "inside_image_vertices": int(image_indices.size),
                    "inside_mask_vertices": int(valid_indices.size),
                    "depth_rejected_vertices": int(instance_depth_rejected),
                    "kept_vertices": int(keep_indices.size),
                    "front_depth": None if front_depth is None else float(front_depth),
                }
            )

        filtered_vertex_data = vertex_data[keep_mask].copy()
        kept_vertices = int(keep_mask.sum())
        inside_image = int(inside_image_mask.sum())
        inside_mask = int(any_mask_projection.sum())
        depth_rejected = int((any_mask_projection & ~keep_mask).sum())

    rebuilt_elements = []
    for element in ply_data.elements:
        element_data = filtered_vertex_data if element.name == "vertex" else element.data
        rebuilt_elements.append(
            PlyElement.describe(element_data, element.name, comments=list(element.comments))
        )

    filtered_ply = PlyData(
        rebuilt_elements,
        text=ply_data.text,
        byte_order=ply_data.byte_order,
        comments=list(ply_data.comments),
        obj_info=list(ply_data.obj_info),
    )
    filtered_ply.write(str(filtered_path))

    expected_width = expected_shape[1]
    expected_height = expected_shape[0]
    if metadata_image_size != (expected_width, expected_height):
        print(
            "Warning: SHARP metadata image size "
            f"{metadata_image_size} does not match uploaded image size {(expected_width, expected_height)}."
        )

    stats = {
        "source_ply_path": ply_path,
        "filtered_ply_path": filtered_path,
        "total_vertices": int(total_vertices),
        "inside_image_vertices": int(inside_image),
        "kept_vertices": int(kept_vertices),
        "inside_mask_vertices": int(inside_mask),
        "depth_rejected_vertices": int(depth_rejected),
        "mask_has_people": bool(any(mask.any() for mask in instance_masks)),
        "foreground_nearest_percentile": float(nearest_percentile),
        "foreground_depth_window": float(depth_window),
        "instances": instance_stats,
    }
    return filtered_path, stats


def display_results(
    np_image: np.ndarray,
    people_mask: np.ndarray,
    artifacts: dict[str, Path],
    segmentation: dict[str, Any],
    filter_stats: dict[str, Any],
) -> None:
    overlay_image = np.array(Image.open(artifacts["overlay_path"]).convert("RGB"))
    inpainted_image = np.array(Image.open(artifacts["inpainted_path"]).convert("RGB"))
    removed_image = np.array(Image.open(artifacts["removed_path"]).convert("RGB"))

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    axes[0].imshow(np_image)
    axes[0].set_title("Original image")
    axes[1].imshow(overlay_image)
    axes[1].set_title("People overlay")
    axes[2].imshow(inpainted_image)
    axes[2].set_title("People removed (FLUX Fill)")
    axes[3].imshow(removed_image)
    axes[3].set_title("People removed (black fill debug)")

    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"Prompt: {segmentation['prompt']}")
    print(f"Inpaint prompt: {INPAINT_PROMPT}")
    print(f"Inpaint guidance scale: {INPAINT_GUIDANCE_SCALE}")
    print(f"Inpaint steps: {INPAINT_NUM_STEPS}")
    print(f"Detected SAM3 instances: {segmentation['num_instances']}")
    print(f"Mask pixels: {int(people_mask.sum())}")
    print(f"Total splats: {filter_stats['total_vertices']}")
    print(f"Projected inside image: {filter_stats['inside_image_vertices']}")
    print(f"Projected inside person masks: {filter_stats['inside_mask_vertices']}")
    print(f"Rejected by depth gate: {filter_stats['depth_rejected_vertices']}")
    print(f"Kept foreground splats: {filter_stats['kept_vertices']}")
    print(
        "Foreground slice used per person: "
        f"percentile={filter_stats['foreground_nearest_percentile']}, depth_window={filter_stats['foreground_depth_window']}"
    )

    for instance_stats in filter_stats["instances"]:
        print(
            f"Instance {instance_stats['instance_index']}: "
            f"mask_pixels={instance_stats['mask_pixels']}, "
            f"inside_mask_vertices={instance_stats['inside_mask_vertices']}, "
            f"depth_rejected={instance_stats['depth_rejected_vertices']}, "
            f"kept={instance_stats['kept_vertices']}, "
            f"front_depth={instance_stats['front_depth']}"
        )


In [ ]:
pil_image, np_image, image_path = load_uploaded_image(OUTPUT_DIR)
print(f"Uploaded image saved to: {image_path}")

segmentation = segment_people_sam3(pil_image, text_prompt=TEXT_PROMPT)
people_mask = segmentation["people_mask"]

artifacts = save_mask_artifacts(np_image, people_mask, OUTPUT_DIR)
artifacts["inpaint_mask_path"] = build_inpaint_mask(artifacts["mask_path"], INPAINT_OUTPUT_DIR)
artifacts["inpainted_path"] = run_flux_fill_inpainting(image_path, artifacts["mask_path"], INPAINT_OUTPUT_DIR, INPAINT_PROMPT)
print(f"Mask saved to: {artifacts['mask_path']}")
print(f"Overlay saved to: {artifacts['overlay_path']}")
print(f"Prepared inpaint mask saved to: {artifacts['inpaint_mask_path']}")
print(f"Inpainted background image saved to: {artifacts['inpainted_path']}")
print(f"Black-fill debug image saved to: {artifacts['removed_path']}")
if not people_mask.any():
    print("No people were segmented. FLUX Fill was skipped, the inpainted background image matches the original, and the filtered PLY will contain zero splats.")

sharp_ply_path = run_sharp(image_path, SHARP_OUTPUT_DIR)
print(f"Full SHARP PLY saved to: {sharp_ply_path}")

filtered_ply_path, filter_stats = filter_ply_with_masks(
    sharp_ply_path,
    segmentation["instance_masks"],
    people_mask.shape,
    OUTPUT_DIR,
)
print(f"Filtered people-only PLY saved to: {filtered_ply_path}")

display_results(np_image, people_mask, artifacts, segmentation, filter_stats)


## Expected outputs

After the run cell finishes, the output directory should contain:

- the uploaded image
- `people_mask.png`
- `people_overlay.png`
- `people_removed_black.png`
- `flux_fill_output/people_mask_inpaint.png`
- `flux_fill_output/people_removed_inpainted.png`
- `sharp_output/<image_stem>.ply`
- `people_only_gaussians.ply`

If SAM3 finds no people, the binary mask will be empty, FLUX.1 Fill will be skipped by copying the original image to the inpainted output path, the per-person filtering loop will have no instances to process, and the filtered PLY will contain zero splats.
